# 03 · Easy · Reading the results

Everything a run returns: scores, the audit trail, findings with evidence, and the
list of what couldn't be seen.

In [ ]:
%pip install /Volumes/main/default/lakehouse_gc/lakehouse_gc-0.4.0-py3-none-any.whl

In [ ]:
from collections import Counter

from lakehouse_gc import run

POLICY_PATH = "/Volumes/main/default/lakehouse_gc/policy.yaml"
result = run(POLICY_PATH, scanners=["uc"], spark=spark)

# Status: succeeded | partial (a scanner failed) | failed
print("status:", result.status.value)

In [ ]:
# Scores: 0-100 or None (nothing assessable). Coverage = share of checks with known evidence.
scores = result.scores
print("governance", scores.governance.score, "coverage", scores.governance.coverage)
print("cost      ", scores.cost.score, "coverage", scores.cost.coverage)
print("overall   ", scores.overall, "grade", scores.grade)

# Audit trail: which checks cost points, and how many.
for d in scores.cost.deductions[:10]:
    print(f"  -{float(d.points):.2f}  {d.resource.resource_id}  {d.rule_id}  ({d.outcome.value})")

In [ ]:
# Findings, with reasons and evidence (observed | inferred | unavailable).
for f in result.findings[:10]:
    print(f.resource.resource_id, f.rule_id, f.recommendation.value, f.confidence)
    print("    why:", " ".join(f.reasons))
    for e in f.evidence:
        print("    evidence:", e.key, e.status.value, e.value, "from", e.source)

In [ ]:
# Quick views: issues per owner, and per recommendation.
issues = [f for f in result.findings if f.recommendation.value not in ("keep", "excluded")]
owners = {r.ref.key: r.owner or "(no owner)" for r in result.scanner_result("uc").resources}
print(Counter(owners.get(f.resource.key, "(unknown)") for f in issues).most_common(10))
print(Counter(f.recommendation.value for f in result.findings))

In [ ]:
# What couldn't be seen (those checks are unknown, never passed), warnings, and failures.
print("limitations:", result.limitations)
print("warnings:   ", result.warnings)
print("errors:     ", result.errors)
print("savings:    ", result.savings.total, "(None = unknown, never a guess)")